# GTEx model building with MOFA-FLEX base

💡 **Environment:** `clamp-analyses`  

# Libraries

In [1]:
library(here)

set.seed(123)

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



In [2]:
library(reticulate)

# Name of the env you expect users to have
env_name <- "clamp-analyses"

if (nzchar(Sys.getenv("RETICULATE_PYTHON"))) {
  message("Using RETICULATE_PYTHON = ", Sys.getenv("RETICULATE_PYTHON"))

} else {
  conda <- Sys.which("conda")

  if (nzchar(conda)) {
    cmd <- sprintf(
      '%s run -n %s python -c "import sys; print(sys.executable)"',
      shQuote(conda), shQuote(env_name)
    )
    py <- tryCatch(system(cmd, intern = TRUE), error = function(e) character(0))

    if (length(py) == 1 && nzchar(py) && file.exists(py)) {
      Sys.setenv(RETICULATE_PYTHON = py)
      message("Auto-set RETICULATE_PYTHON = ", py)
    } else {
      message("Could not resolve env python via conda. Falling back to Sys.which('python').")
      Sys.setenv(RETICULATE_PYTHON = Sys.which("python"))
    }

  } else {
    message("conda not found on PATH. Falling back to Sys.which('python').")
    Sys.setenv(RETICULATE_PYTHON = Sys.which("python"))
  }
}

py_config()


Could not resolve env python via conda. Falling back to Sys.which('python').



python:         /home/msubirana/miniconda3/envs/clamp-analyses/bin/python
libpython:      /home/msubirana/miniconda3/envs/clamp-analyses/lib/libpython3.11.so
pythonhome:     /home/msubirana/miniconda3/envs/clamp-analyses:/home/msubirana/miniconda3/envs/clamp-analyses
version:        3.11.14 | packaged by conda-forge | (main, Jan 26 2026, 23:48:32) [GCC 14.3.0]
numpy:          /home/msubirana/miniconda3/envs/clamp-analyses/lib/python3.11/site-packages/numpy
numpy_version:  2.3.5

NOTE: Python version was forced by RETICULATE_PYTHON

In [3]:
mfl <- import("mofaflex", delay_load = FALSE)
ad  <- import("anndata",  delay_load = FALSE)
np  <- import("numpy",    delay_load = FALSE)
pd  <- import("pandas",   delay_load = FALSE)

# Input

In [4]:
gtex_rds <- here("output/gtex/df_gtex_fbm_filt.rds")
k_rds    <- here("output/gtex/CLAMP_K_gtex.rds")

stopifnot(file.exists(gtex_rds), file.exists(k_rds))

gtex_data <- readRDS(gtex_rds)   # genes x samples
K <- readRDS(k_rds)              # scalar

dim(gtex_data)
K

stopifnot(is.numeric(K), length(K) == 1)
stopifnot(!is.null(rownames(gtex_data)), !is.null(colnames(gtex_data)))

[1] 21613 17382

[1] 578
attr(,"limit")
[1] 1.530897

In [5]:
# AnnData samples x genes
X <- t(as.matrix(gtex_data))

stopifnot(K <= min(nrow(X), ncol(X)))

# lower memory than float64
X_np <- np$array(X, dtype = "float32")
adata <- ad$AnnData(X_np)

# preserve names
adata$obs_names <- pd$Index(colnames(gtex_data))  # samples
adata$var_names <- pd$Index(rownames(gtex_data))  # genes

# MOFA-FLEX expects nested dict: group -> view -> AnnData
data_list <- dict(group_1 = dict(view_1 = adata))

# MOFA-FLEX

In [6]:
model <- mfl$MOFAFLEX(
  data_list,
  mfl$DataOptions(
    scale_per_group = FALSE, # already z-scored
    plot_data_overview = FALSE,
    remove_constant_features = TRUE
  ),
  mfl$ModelOptions(
    n_factors = as.integer(K),
    likelihoods = "Normal",
    weight_prior = "Laplace"
  ),
  mfl$TrainingOptions(
    seed = 123L,
    max_epochs = 2000L,
    batch_size = 1000L
  )
)

In [ ]:
# extract factors Z (samples x K) then convert to B (K x samples)
Z_dict <- model$get_factors(return_type = "numpy", ordered = FALSE)
Z <- py_to_r(Z_dict[["group_1"]])       # samples x K

B <- t(Z)                                 # K x samples
rownames(B) <- paste0("LV", seq_len(nrow(B)))
colnames(B) <- colnames(gtex_data)

# write outputs
output_dir <- here("output/gtex/MOFA_FLEX_BASE")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

write.csv(
  B,
  file = file.path(output_dir, "gtex_B.csv"),
  quote = FALSE
)

# save model
py_save_object(model, file.path(output_dir, "mofaflex_model.pkl"))